# Taxonomía: árbol de especies desde GBIF

Construye el árbol taxonómico completo (reino → filo → clase → orden → familia → género → especie) para todas las especies que ya conocemos, ya sea porque están en `BASE_DATOS_GENERAL.xlsx` (lista oficial de Campus Lab) o porque ya aparecen en la tabla `especies` de la BD.

**Idea tomada de LinNeo** (proyecto personal de Remi: https://github.com/RemiH06/LinNeo), pero simplificada: aquí no se monta Neo4j, se guarda el mismo árbol padre-hijo como una tabla normal de SQLite (`taxonomia_nodos`, con `gbif_key_padre` apuntando a su propio `gbif_key`), suficiente para el tamaño de este catálogo (cientos de especies, no todo el árbol de la vida).

**Fuente:** [GBIF Species API](https://www.gbif.org/developer/species) — `species/match`, abierta, sin API key, sin cuota diaria conocida.

**Insumo:** `../../experimentos/bd/campuslab_volumen.db` (tablas `especies` y `referencia_nombres_comunes`).
**Salida:** tabla nueva `taxonomia_nodos`, columna nueva `especies.gbif_key` (+ `gbif_match_type`, `gbif_confidence` de auditoría), y un Excel con los nombres que no dieron un match confiable para revisión aparte.

**No toca `fotos` ni `estado` de ninguna foto** — esto es solo metadata de especie, no identificación de imágenes.

In [1]:
import json
import sqlite3
import time
from pathlib import Path

import pandas as pd
import requests

RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")  # la BD real, sin mover ni duplicar
con = sqlite3.connect(RUTA_BD)
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

Base de datos: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\experimentos\bd\campuslab_volumen.db (existe)


## 1. Esquema: tabla de nodos taxonómicos + columnas de enlace en `especies`

In [2]:
con.execute("""
    CREATE TABLE IF NOT EXISTS taxonomia_nodos (
        gbif_key INTEGER PRIMARY KEY,
        rango TEXT NOT NULL,              -- KINGDOM / PHYLUM / CLASS / ORDER / FAMILY / GENUS / SPECIES
        nombre_cientifico TEXT NOT NULL,
        gbif_key_padre INTEGER REFERENCES taxonomia_nodos(gbif_key)
    )
""")

cols_especies = [r[1] for r in con.execute("PRAGMA table_info(especies)")]
for col, tipo in [("gbif_key", "INTEGER REFERENCES taxonomia_nodos(gbif_key)"),
                   ("gbif_match_type", "TEXT"),
                   ("gbif_confidence", "INTEGER")]:
    if col not in cols_especies:
        con.execute(f"ALTER TABLE especies ADD COLUMN {col} {tipo}")
con.commit()
print("Esquema listo: taxonomia_nodos, especies.gbif_key/gbif_match_type/gbif_confidence")

Esquema listo: taxonomia_nodos, especies.gbif_key/gbif_match_type/gbif_confidence


## 2. Nombres a mapear: lista oficial + lo que ya está en `especies`

Union de `referencia_nombres_comunes.nombre_cientifico` (la lista oficial, `BASE_DATOS_GENERAL.xlsx`) y `especies.nombre_cientifico` (lo que ya se ha sugerido/confirmado, aunque no todo ahí sea un binomio real todavía). Se salta lo que ya tiene `gbif_key` para que esto sea resumible.

In [3]:
oficiales = {r[0] for r in con.execute(
    "SELECT DISTINCT nombre_cientifico FROM referencia_nombres_comunes WHERE nombre_cientifico IS NOT NULL"
)}
en_especies = {r[0] for r in con.execute(
    "SELECT DISTINCT nombre_cientifico FROM especies WHERE nombre_cientifico IS NOT NULL"
)}
ya_mapeados = {r[0] for r in con.execute(
    "SELECT DISTINCT nombre_cientifico FROM especies WHERE gbif_key IS NOT NULL"
)}

objetivo = sorted((oficiales | en_especies) - ya_mapeados)
print(f"Nombres oficiales: {len(oficiales)}")
print(f"Nombres ya en especies: {len(en_especies)}")
print(f"Ya mapeados en corridas anteriores: {len(ya_mapeados)}")
print(f"Pendientes de mapear ahora: {len(objetivo)}")

Nombres oficiales: 430
Nombres ya en especies: 176
Ya mapeados en corridas anteriores: 149
Pendientes de mapear ahora: 319


## 3. Función de match contra GBIF

Un solo llamado por nombre trae toda la cadena de ancestros (reino...género) con su propio `key` de GBIF, no hace falta otro endpoint para subir el árbol.

In [4]:
import re

RANGOS = [
    ("KINGDOM", "kingdom", "kingdomKey"),
    ("PHYLUM", "phylum", "phylumKey"),
    ("CLASS", "class", "classKey"),
    ("ORDER", "order", "orderKey"),
    ("FAMILY", "family", "familyKey"),
    ("GENUS", "genus", "genusKey"),
]

# "Genus sp." es la convencion estandar para "especie de este genero, sin identificar
# a nivel de especie" - NO es un error de captura. Pero preguntarle a GBIF el nombre
# completo con "sp." a veces regresa NONE (paso con "Erythrina sp.", "Ficus sp.",
# "Pinus sp.", etc.) aunque el genero SI exista y este bien escrito - GBIF es
# inconsistente con el sufijo literal. Se quita "sp."/"sp" antes de preguntar y se
# pide un match a nivel genero en su lugar (matchType EXACT/FUZZY/HIGHERRANK, el
# genero solo casi siempre regresa HIGHERRANK porque /match esta pensado para
# binomios completos, no es un error).
PATRON_SP_INDETERMINADA = re.compile(r"\s+sp\.?$", re.IGNORECASE)

def gbif_match(nombre_cientifico):
    """Llama a GBIF species/match. Devuelve el JSON crudo o None si truena la conexion."""
    resp = requests.get(
        "https://api.gbif.org/v1/species/match",
        params={"name": nombre_cientifico, "strict": False},
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()

def resolver_nombre(nombre_cientifico):
    """Detecta 'Genero sp.' y pregunta solo por el genero en ese caso.
    Devuelve (cuerpo_json, es_indeterminada)."""
    es_indeterminada = bool(PATRON_SP_INDETERMINADA.search(nombre_cientifico))
    nombre_consulta = PATRON_SP_INDETERMINADA.sub("", nombre_cientifico) if es_indeterminada else nombre_cientifico
    return gbif_match(nombre_consulta), es_indeterminada

def es_match_confiable(cuerpo, es_indeterminada=False):
    """EXACT o FUZZY con rango de especie, o (si el nombre original era 'Genero sp.')
    EXACT/FUZZY/HIGHERRANK con rango GENUS - /match regresa HIGHERRANK para consultas de
    solo-genero casi siempre, no es una señal de mala calidad en ese caso especifico.
    NONE/HIGHERRANK a nivel familia o mas arriba (fuera de este caso) si significa
    'no encontre esa especie'."""
    if not cuerpo:
        return False
    tipos_validos = ("EXACT", "FUZZY", "HIGHERRANK") if es_indeterminada else ("EXACT", "FUZZY")
    if cuerpo.get("matchType") not in tipos_validos:
        return False
    rangos_validos = ("GENUS",) if es_indeterminada else ("SPECIES", "SUBSPECIES", "VARIETY", "GENUS")
    if cuerpo.get("rank") not in rangos_validos:
        return False
    return (cuerpo.get("confidence") or 0) >= 80

def guardar_nodos(cuerpo):
    """Inserta (si hace falta) cada rango de la cadena reino->genero->especie, encadenando padres."""
    padre_key = None
    for rango, campo_nombre, campo_key in RANGOS:
        key = cuerpo.get(campo_key)
        nombre = cuerpo.get(campo_nombre)
        if key is None or nombre is None:
            continue
        con.execute(
            "INSERT OR IGNORE INTO taxonomia_nodos (gbif_key, rango, nombre_cientifico, gbif_key_padre) VALUES (?,?,?,?)",
            (key, rango, nombre, padre_key),
        )
        padre_key = key

    # Nodo de especie (o el rango que haya devuelto GBIF como "usageKey"), colgado del genero.
    # Para un match de solo-genero (caso "Genero sp."), usageKey == genusKey: el INSERT OR
    # IGNORE de abajo no duplica nada (misma clave primaria), y especie_key queda apuntando
    # correctamente al nodo GENUS ya insertado arriba.
    especie_key = cuerpo.get("speciesKey") or cuerpo.get("usageKey")
    nombre_especie = cuerpo.get("species") or cuerpo.get("canonicalName")
    if especie_key is not None and nombre_especie is not None:
        con.execute(
            "INSERT OR IGNORE INTO taxonomia_nodos (gbif_key, rango, nombre_cientifico, gbif_key_padre) VALUES (?,?,?,?)",
            (especie_key, "SPECIES", nombre_especie, padre_key),
        )
        return especie_key
    return padre_key  # no llego a especie, se queda colgado del genero

## 4. Correr contra todos los nombres pendientes

Sin cuota diaria conocida en GBIF, así que se corre todo de un jalón (con una pausa chica entre llamadas por cortesía). Cada nombre se intenta aparte con try/except: si uno falla no tumba el resto, y es resumible (`objetivo` ya excluye lo que quedó bien la vez pasada).

In [5]:
resultados = []
for nombre in objetivo:
    try:
        cuerpo, es_indeterminada = resolver_nombre(nombre)
        confiable = es_match_confiable(cuerpo, es_indeterminada)
        especie_key = guardar_nodos(cuerpo) if confiable else None
        if especie_key is not None:
            con.execute(
                "UPDATE especies SET gbif_key=?, gbif_match_type=?, gbif_confidence=? WHERE nombre_cientifico=?",
                (especie_key, cuerpo.get("matchType"), cuerpo.get("confidence"), nombre),
            )
        resultados.append({
            "nombre": nombre, "confiable": confiable, "es_indeterminada": es_indeterminada,
            "matchType": cuerpo.get("matchType"), "rank": cuerpo.get("rank"),
            "confidence": cuerpo.get("confidence"), "canonicalName": cuerpo.get("canonicalName"),
            "error": None,
        })
    except Exception as e:
        resultados.append({"nombre": nombre, "confiable": False, "error": str(e)})
    time.sleep(0.1)
con.commit()

df_resultado = pd.DataFrame(resultados)
print(f"Procesados: {len(df_resultado)}")
print(f"Match confiable: {df_resultado['confiable'].sum()}")
print(f"Sin match confiable: {(~df_resultado['confiable']).sum()}")

Procesados: 319
Match confiable: 273
Sin match confiable: 46


## 5. Reporte: nombres sin match confiable, y resumen del árbol

In [6]:
sin_match = df_resultado[~df_resultado["confiable"]]
ruta_sin_match = Path("../salida/Taxonomia_Sin_Match_GBIF.xlsx")
ruta_sin_match.parent.mkdir(exist_ok=True)
sin_match.to_excel(ruta_sin_match, index=False)
print(f"Guardado: {ruta_sin_match} ({len(sin_match)} nombres sin match confiable, para revisar aparte)")

print()
print("--- Árbol construido hasta ahora ---")
for rango, n in con.execute("SELECT rango, COUNT(*) FROM taxonomia_nodos GROUP BY rango ORDER BY COUNT(*) DESC"):
    print(f"  {rango}: {n}")

print()
print("--- Especies con taxonomia enlazada ---")
con_taxon = con.execute("SELECT COUNT(*) FROM especies WHERE gbif_key IS NOT NULL").fetchone()[0]
total = con.execute("SELECT COUNT(*) FROM especies").fetchone()[0]
print(f"  {con_taxon} / {total}")

Guardado: ..\salida\Taxonomia_Sin_Match_GBIF.xlsx (46 nombres sin match confiable, para revisar aparte)

--- Árbol construido hasta ahora ---
  SPECIES: 408
  GENUS: 272
  FAMILY: 102
  ORDER: 41
  CLASS: 5
  PHYLUM: 2
  KINGDOM: 2

--- Especies con taxonomia enlazada ---
  149 / 176


## 6. Para los que no dieron match: buscar el nombre científico por nombre común (Wikidata)

Muchos de los "sin match" no son errores de GBIF — son nombres comunes en español que quedaron
en la columna de nombre científico de `BASE_DATOS_GENERAL.xlsx` (ej. "Cedro rojo", en vez de
*Cedrela odorata*). GBIF espera un binomio latino, así que nunca los iba a encontrar.

Wikidata tiene el campo estructurado `P225` ("taxon name") pensado exactamente para esto, y
reconoce alias en español (confirmado: "Cedro rojo" → *Cedrela odorata*, "Zapote negro" →
*Diospyros nigra*). **No se confía en Wikidata solo**: cualquier candidato se vuelve a pasar por
`resolver_nombre` + `es_match_confiable` de GBIF antes de guardar nada — mismo filtro que todo lo
demás en este notebook.

**No renombra `especies.nombre_cientifico`** todavía, solo enlaza `gbif_key` — si el nombre
científico real ya existe como otra fila (ej. alguien ya confirmó *Cedrela odorata* por
Pl@ntNet), renombrar crearía un duplicado. Fusionar esas filas (reapuntando `fotos.especie_id`)
queda como paso aparte, después de ver cuántos casos así hay de verdad.

In [7]:
HEADERS_WIKI = {"User-Agent": "CampusLabITESO-TaxonomiaGBIF/1.0 (proyecto Campus Lab ITESO, uso interno)"}
# Wikidata/Wikimedia rechaza peticiones sin User-Agent (responden vacio, no un error claro).

def buscar_nombre_cientifico_wikidata(nombre_comun, intentos=3):
    """Busca el nombre comun en Wikidata (es) y regresa el 'taxon name' (P225) del
    primer resultado, si existe. None si no hay resultado o no tiene ese campo.
    Reintenta en errores de red/throttling (no en 'no encontrado', eso es una
    respuesta valida, no una falla)."""
    for intento in range(intentos):
        try:
            r = requests.get("https://www.wikidata.org/w/api.php", params={
                "action": "wbsearchentities", "search": nombre_comun, "language": "es",
                "type": "item", "limit": 1, "format": "json",
            }, timeout=10, headers=HEADERS_WIKI)
            r.raise_for_status()
            resultados = r.json().get("search", [])
            if not resultados:
                return None
            qid = resultados[0]["id"]
            r2 = requests.get("https://www.wikidata.org/w/api.php", params={
                "action": "wbgetclaims", "entity": qid, "property": "P225", "format": "json",
            }, timeout=10, headers=HEADERS_WIKI)
            r2.raise_for_status()
            claims = r2.json().get("claims", {}).get("P225", [])
            if not claims:
                return None
            return claims[0]["mainsnak"]["datavalue"]["value"]
        except (requests.RequestException, ValueError, KeyError, IndexError):
            if intento == intentos - 1:
                return None
            time.sleep(1.5 * (intento + 1))  # backoff: 1.5s, 3s
    return None


resultados_wikidata = []
for nombre in sin_match["nombre"]:
    candidato = buscar_nombre_cientifico_wikidata(nombre)
    fila = {"nombre_original": nombre, "candidato_wikidata": candidato,
            "validado_gbif": False, "ya_existe_en_especies": False}

    if candidato:
        cuerpo, es_indeterminada = resolver_nombre(candidato)
        fila["validado_gbif"] = es_match_confiable(cuerpo, es_indeterminada)
        if fila["validado_gbif"]:
            ya_existe = con.execute(
                "SELECT id FROM especies WHERE nombre_cientifico=? AND nombre_cientifico != ?",
                (candidato, nombre),
            ).fetchone()
            fila["ya_existe_en_especies"] = ya_existe is not None
            # Se enlaza gbif_key siempre que sea confiable (aditivo, no crea duplicados).
            # NO se renombra nombre_cientifico aqui: si ya_existe_en_especies es True hace
            # falta fusionar las filas (reapuntar fotos.especie_id) a mano, no de un jalon.
            especie_key = guardar_nodos(cuerpo)
            con.execute(
                "UPDATE especies SET gbif_key=?, gbif_match_type=?, gbif_confidence=? WHERE nombre_cientifico=?",
                (especie_key, cuerpo.get("matchType"), cuerpo.get("confidence"), nombre),
            )
    resultados_wikidata.append(fila)
    time.sleep(0.5)  # Wikidata + GBIF, dos llamadas por nombre - cortesia extra, evita throttling
con.commit()

df_wikidata = pd.DataFrame(resultados_wikidata)
ruta_wikidata = Path("../salida/Taxonomia_Candidatos_Wikidata.xlsx")
df_wikidata.to_excel(ruta_wikidata, index=False)

print(f"Candidato encontrado en Wikidata: {df_wikidata['candidato_wikidata'].notna().sum()} / {len(df_wikidata)}")
print(f"Validado tambien por GBIF:        {df_wikidata['validado_gbif'].sum()}")
print(f"De esos, ya existe otra fila con ese nombre (necesita fusion manual): {df_wikidata['ya_existe_en_especies'].sum()}")
print(f"\nGuardado: {ruta_wikidata}")
print()
print(df_wikidata.to_string(index=False))

Candidato encontrado en Wikidata: 10 / 46
Validado tambien por GBIF:        2
De esos, ya existe otra fila con ese nombre (necesita fusion manual): 0

Guardado: ..\salida\Taxonomia_Candidatos_Wikidata.xlsx

     nombre_original   candidato_wikidata  validado_gbif  ya_existe_en_especies
   Acacia sieberiana    Acacia sieberiana          False                  False
       Agave cantala        Agave cantala          False                  False
        Amate peludo                 None          False                  False
      Bucida buceras       Bucida buceras          False                  False
    Calistemo llorón                 None          False                  False
       Colorín flama                 None          False                  False
 Crataegus pubescens                 None          False                  False
      Datura arborea                 None          False                  False
    Diospyros digyna     Diospyros digyna          False                 